# Lab Kit L2 · Drive the arm

**Video:** Lab Kit L2 on [Foundation Models for Robotics](https://www.youtube.com/@FoundationModelsForRobotics)
**You need:** Lab Kit L1. About 20 minutes.

In L1 you told the arm how to bend each joint. Usually you want the opposite: *put the hand there* and let the computer work out the joints. That's inverse kinematics (IK). Track A (A3, A4) explains how it works inside; here you learn to use it.

In [ ]:
# Setup: works in Google Colab and locally (after `pip install -e .` in the repo root)
import os, sys
if "google.colab" in sys.modules:
    os.environ["MUJOCO_GL"] = "egl"
    if not os.path.exists("fmr-robot-learning-labs"):
        !git clone -q https://github.com/kandapagari/fmr-robot-learning-labs.git
    %cd fmr-robot-learning-labs
    !pip install -q -e .
import numpy as np, matplotlib.pyplot as plt, mediapy as media, mujoco
import fmr6


def show_video(frames, fps=30):
    """Play frames inline. Without ffmpeg (some local installs) show a strip of stills instead."""
    try:
        media.show_video(frames, fps=fps)
    except RuntimeError:
        media.show_images(frames[:: max(1, len(frames) // 5)], columns=5)

## 1 · Put the hand at a point

`robot.ik(target)` returns joint angles that put the TCP at `target` (metres) with the hand pointing straight down. It starts from the current pose and improves it step by step.

In [ ]:
robot = fmr6.Robot("table", arm="fr3")
target = np.array([0.60, 0.10, 0.45])
q = robot.ik(target)
pos, R = robot.tcp(q)
print("asked for", target, " got", pos.round(4), f" error {np.linalg.norm(pos - target) * 1000:.2f} mm")
media.show_image(robot.render(560, 360, camera={"lookat": [0.66, -0.05, 0.42], "distance": 1.2, "azimuth": 150, "elevation": -28}))

## 2 · Two ways to get from A to B

Moving from pose A to pose B, you can interpolate the **joint angles** (each joint turns at a steady rate) or the **hand position** (solve IK at every point along a straight line). Both arrive at B. The red line shows where the fingertips went.

In [ ]:
A, B = np.array([0.55, -0.20, 0.45]), np.array([0.55, 0.20, 0.45])
qA = robot.ik(A, q0=robot.ready); qB = robot.ik(B, q0=qA)

def run(kind, n=40):
    tips, frames, q = [], [], qA
    for s in np.linspace(0, 1, n):
        q = qA + s * (qB - qA) if kind == "joint" else robot.ik(A + s * (B - A), q0=q)
        robot.set_q(q)
        tips.append(robot.tcp()[0])
        frames.append(robot.render(400, 280, camera={"lookat": [0.66, -0.05, 0.42], "distance": 1.2, "azimuth": 150, "elevation": -28}, trail=np.array(tips), guide=np.array([A, B])))
    return np.array(tips), frames

tips_j, frames_j = run("joint")
tips_c, frames_c = run("cartesian")
print(f"joint interpolation: tip strays up to {np.abs(tips_j[:, 0] - A[0]).max() * 1000:.0f} mm off the straight line")
print(f"cartesian:           {np.abs(tips_c[:, 0] - A[0]).max() * 1000:.1f} mm")
media.show_images([frames_j[-1], frames_c[-1]], titles=["joint interpolation", "straight line (IK every step)"])

Joint interpolation is simple and smooth for the motors, but the hand sweeps an arc. If something is in the way, or you're wiping a table, you want the straight line. Many learned policies output hand positions for this reason.

## 3 · Point the hand somewhere else

`tool_dir` sets where the hand points; `grip_dir` sets the direction the fingers close along. Both are directions in the world frame.

In [ ]:
poses = [
    ("down", dict(tool_dir=(0, 0, -1))),
    ("down, fingers across y", dict(tool_dir=(0, 0, -1), grip_dir=(0, 1, 0))),
    ("tilted forward 45°", dict(tool_dir=(1, 0, -1))),
]
pics = []
for title, kw in poses:
    q = robot.ik([0.55, 0.0, 0.50], q0=robot.ready, **kw)
    pics.append(robot.render(320, 240, camera={"lookat": [0.66, -0.05, 0.42], "distance": 1.2, "azimuth": 150, "elevation": -28}))
media.show_images(pics, titles=[p[0] for p in poses])

## 4 · When IK can't do it

Ask for a point the arm can't reach and IK does its best, then stops with an error. Always check the error before you trust the joint angles.

In [ ]:
for x in (0.60, 0.70, 0.75, 0.80, 0.90):
    t = np.array([x, 0.0, 0.45])
    q = robot.ik(t, q0=robot.ready)
    err = np.linalg.norm(robot.tcp(q)[0] - t)
    print(f"x = {x:.2f} m → error {err * 1000:7.1f} mm", "" if err < 1e-3 else "  ← out of reach")

## 5 · Cameras and saving a video

`camera=` takes `lookat` (the point at the centre of the image), `distance`, `azimuth` (degrees around the vertical) and `elevation` (degrees up/down; negative looks down). Orbit the camera while the arm draws its straight line, then save an MP4 you can share.

In [ ]:
orbit = []
for i, f in enumerate(frames_c + frames_c[::-1]):
    robot.set_q(robot.ik(tips_c[min(i, 2 * len(tips_c) - 1 - i)], q0=robot.q()))
    cam = {"lookat": [0.55, 0.0, 0.45], "distance": 1.6, "azimuth": 120 + 1.5 * i, "elevation": -25}
    orbit.append(robot.render(480, 320, camera=cam, trail=tips_c))
show_video(orbit, fps=30)
try:
    media.write_video("my_first_robot_video.mp4", orbit, fps=30)
    print("saved my_first_robot_video.mp4 (in Colab: Files panel on the left → download)")
except RuntimeError:
    print("no ffmpeg here, so no MP4; the frames above are still fine")

## 6 · Try it

1. Make the hand draw a 20 cm square on the table, 5 cm above the surface (z = 0.43). Plot the four corners with `guide=`.
2. Repeat section 2 with `fmr6.Robot("table", arm="fmr6")`. Does joint interpolation stray more or less?
3. Find the furthest point straight ahead (y = 0, z = 0.45) the FR3 can reach with the hand pointing down, to the nearest centimetre.
4. The FR3 has seven joints and the target only fixes six numbers (3 position, 3 orientation). Start IK from two different `q0` and compare the joint angles you get. Same hand pose, different arm?

**Next:** L3 runs the real physics, so motors take time, objects fall and fingers can grip.

In [ ]:
# your experiments here